# Split Comparison — Leakage in the Original Random Split (not a benchmark result)

The original paper split addresses at random. Wallets in the same gas provision tree (the wallets that share the
root of an unlabeled provision chain) share their provider and tree feature values, so under a random split
test wallets can match training wallets exactly on those features (relational leakage). Reviewer 2 raised
this.

This notebook measures that leakage. It trains the same models, with the same hyperparameters (selected on
the group split by `02`), on the original address-level random split, using the same code as the model
notebooks (`sp.fit_xgb`, `sp.fit_lgbm`, `sp.fit_lr`, `sp.select_blend_weight`, `sp.evaluate`), and compares
the results with the group-split results of `03`–`06`. The reported results are the group-split ones; the
random split is run only here.

Runs last: it reads `03`–`06`'s results and test predictions and checks that they trace to this commit.

## Setup

In [1]:
import warnings
import pandas as pd
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

GROUP_NBS = ['03_xgboost_sybil', '04_lightgbm_sybil', '05_logistic_regression_sybil', '06_cross_ensemble_sybil']
prov = sp.provenance(GROUP_NBS)
print(prov.to_string(index=False))
assert prov['dependencies_unchanged'].all(), 'Group-split results were produced by different code: rerun them'

                    notebook  commit  dependencies_unchanged
            03_xgboost_sybil 190c074                    True
           04_lightgbm_sybil 4e3ec91                    True
05_logistic_regression_sybil 02f3113                    True
     06_cross_ensemble_sybil 189f0d5                    True


## 1. Relational leakage under each split

In [2]:
df, _, _ = sp.build_master_df('./data', verbose=False)
S = {m: sp.make_splits(df, sp.FEATS, method=m, seed=sp.SEED) for m in ['random', 'group']}
leak = {}
for m in ['random', 'group']:
    leak[m] = sp.leakage_report(df, S[m]); print()
print(pd.DataFrame(leak).T[['test_rows', 'test_sybils', 'test_rows_sharing_tree_with_train',
                            'test_sybils_sharing_tree_with_train_sybil']].to_string())

Split method: random
Row overlap (train/val, train/test, val/test): 0, 0, 0
Test addresses sharing a gas provision tree with a train address: 19,884 / 130,436
Test Sybils sharing a gas provision tree with a train Sybil:     599 / 5,463

Split method: group
Row overlap (train/val, train/test, val/test): 0, 0, 0
Test addresses sharing a gas provision tree with a train address: 0 / 130,435
Test Sybils sharing a gas provision tree with a train Sybil:     0 / 5,463

       test_rows test_sybils test_rows_sharing_tree_with_train test_sybils_sharing_tree_with_train_sybil
random    130436        5463                             19884                                       599
group     130435        5463                                 0                                         0


## 2. Train the same models on the random split

Same hyperparameters as the group-split models; thresholds and the blend weight chosen on the random
split's own validation partition.

In [3]:
XGB_SELECTED, LGBM_SELECTED = sp.load_selected_params()
Sr = S['random']
xgb = sp.fit_xgb(Sr, XGB_SELECTED)
lgbm = sp.fit_lgbm(Sr, LGBM_SELECTED)
lr = sp.fit_lr(Sr)
w, _ = sp.select_blend_weight(Sr['y_val'], xgb['val'], lgbm['val'])
ens = {part: w * xgb[part] + (1 - w) * lgbm[part] for part in ['val', 'test']}
print(f'Blend weight on the random split: XGB {w:.2f}')
RANDOM = {name: sp.evaluate(name, Sr['y_val'], p['val'], Sr['y_test'], p['test']) for name, p in [
    ('XGBoost (tuned, 3-seed)', xgb), ('LightGBM (tuned, 3-seed)', lgbm),
    ('Logistic Regression (C=0.01, L1)', lr), ('Cross-Ensemble', ens)]}

  XGBoost seed=42  rounds=1753  76.3s
  XGBoost seed=123  rounds=1694  69.8s
  XGBoost seed=456  rounds=1719  70.1s
  LightGBM seed=42  rounds=1340  53.5s
  LightGBM seed=123  rounds=1345  52.3s
  LightGBM seed=456  rounds=1279  48.3s
Blend weight on the random split: XGB 0.06


## 3. Random vs group split (test set)

In [4]:
GROUP = {}
for nb in GROUP_NBS:
    for r in sp.load_results(nb)['models']:
        GROUP.setdefault(r['name'], r)          # 06 repeats 03/04's single models; they are identical
rows = []
for name, r in RANDOM.items():
    g = GROUP[name]
    row = {'Model': name}
    for k in ['f1', 'auroc', 'ap']:
        row[(k, 'random')], row[(k, 'group')], row[(k, 'Δ')] = r[k], g[k], g[k] - r[k]
    rows.append(row)
comp = pd.DataFrame(rows).set_index('Model')
comp.columns = pd.MultiIndex.from_tuples(comp.columns)
print(comp.round(4).to_string())

                                      f1                   auroc                      ap                
                                  random   group       Δ  random   group       Δ  random   group       Δ
Model                                                                                                   
XGBoost (tuned, 3-seed)           0.7410  0.7200 -0.0210  0.9742  0.9657 -0.0084  0.7976  0.7645 -0.0331
LightGBM (tuned, 3-seed)          0.7418  0.7188 -0.0230  0.9756  0.9683 -0.0074  0.8024  0.7696 -0.0328
Logistic Regression (C=0.01, L1)  0.2335  0.2412  0.0077  0.8329  0.8368  0.0039  0.1601  0.1656  0.0056
Cross-Ensemble                    0.7426  0.7173 -0.0253  0.9757  0.9683 -0.0073  0.8026  0.7699 -0.0327


## 4. Where the leakage sits: F1 by taxonomy category

Share of each taxonomy category in multi-wallet gas provision trees, then test F1 under each split. IxL wallets
are each their own tree unless they fund other wallets.

In [5]:
PROB = {'XGBoost (tuned, 3-seed)': ('03_xgboost_sybil', 'prob_xgb', xgb),
        'LightGBM (tuned, 3-seed)': ('04_lightgbm_sybil', 'prob_lgbm', lgbm)}
rows = []
for name, (nb, col, m) in PROB.items():
    pg = sp.load_predictions(nb, 'test')
    fr = sp.metrics_by_category(Sr['y_test'], m['test'], df.loc[Sr['idx_test'], 'category'], RANDOM[name]['threshold'])
    fg = sp.metrics_by_category(pg['y'], pg[col], pg['category'], GROUP[name]['threshold'])
    t = fr[['category', 'f1']].merge(fg[['category', 'f1']], on='category', suffixes=('_random', '_group'))
    rows.append(t.assign(Model=name, delta=t['f1_group'] - t['f1_random']))
by_cat = pd.concat(rows, ignore_index=True)[['Model', 'category', 'f1_random', 'f1_group', 'delta']]
size = df['provision_tree'].map(df['provision_tree'].value_counts())
print((size > 1).groupby(df['category']).agg(['sum', 'size', 'mean']).rename(
    columns={'sum': 'in_multi_wallet_trees', 'size': 'wallets', 'mean': 'share'}).round(3).to_string(), '\n')
print(by_cat.round(4).to_string(index=False))

             in_multi_wallet_trees  wallets  share
category                                          
IxE                          38642    93061  0.415
IxI                          34390    34391  1.000
IxL                          12886   306658  0.042
No provider                     56      676  0.083 

                   Model    category  f1_random  f1_group   delta
 XGBoost (tuned, 3-seed)         IxE     0.5316    0.2645 -0.2671
 XGBoost (tuned, 3-seed)         IxI     0.8467    0.5340 -0.3127
 XGBoost (tuned, 3-seed)         IxL     0.7382    0.7448  0.0066
 XGBoost (tuned, 3-seed) No provider     0.5455    0.8750  0.3295
LightGBM (tuned, 3-seed)         IxE     0.5382    0.2885 -0.2498
LightGBM (tuned, 3-seed)         IxI     0.8497    0.4716 -0.3781
LightGBM (tuned, 3-seed)         IxL     0.7388    0.7496  0.0109
LightGBM (tuned, 3-seed) No provider     0.5455    0.8750  0.3295


## Save results

In [6]:
sp.save_results(list(RANDOM.values()), '11_split_comparison', leakage=leak, extra=dict(
    random_split_xgb_weight=w, comparison={n: {k: comp.loc[n, k].to_dict() for k in ['f1', 'auroc', 'ap']} for n in comp.index},
    f1_by_category=by_cat.to_dict('records'), params=dict(xgb=XGB_SELECTED, lgbm=LGBM_SELECTED)))

Saved results/11_split_comparison.json
